# 05 - Azure Infra Setup (Search + Foundry)

Goal: Deploy core Azure AI infrastructure (Search, Foundry, Storage, RBAC) using a unified Bicep template, then create search indices with Python.

**What this notebook does:**
1. Deploys consolidated Bicep template to create:
   - Azure AI Search service with managed identity
   - Azure AI Foundry with project and model deployments
   - Shared storage account with separate containers for search and foundry
   - RBAC assignments for all services
2. Creates search indices using Python SDK
3. Seeds sample documents (2-3 per index)

**Prerequisites:**
- `.env` file with `AGENT_BLUEPRINT_PRINCIPAL_ID` (other vars have sensible defaults)
- Admin permissions in your Azure subscription (you'll be prompted to log in interactively)

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`...

In [1]:
import os
import json
import subprocess
from dotenv import load_dotenv
import utils
from azure.identity import InteractiveBrowserCredential

load_dotenv()

# Fix PATH for Azure CLI - common installation locations
az_paths = [
    '/usr/local/bin',
    '/opt/homebrew/bin',
    '/usr/bin',
    os.path.expanduser('~/bin')
]
for path in az_paths:
    if path not in os.environ['PATH']:
        os.environ['PATH'] = f"{path}:{os.environ['PATH']}"

subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
# Load configuration from .env
resource_group_name = os.getenv('AZURE_RESOURCE_GROUP', 'rg-agent-identity-sandbox')
resource_group_location = os.getenv('AZURE_LOCATION', 'eastus')
search_service_name = os.getenv('AZURE_SEARCH_SERVICE_NAME', '')
storage_account_name = os.getenv('AZURE_STORAGE_ACCOUNT_NAME', '')
blueprint_principal_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID')

# Initialize interactive browser credential for admin login
# This gets a token that az CLI can use
credential = InteractiveBrowserCredential()

# Validate required variables
required = {
    'AGENT_BLUEPRINT_PRINCIPAL_ID': blueprint_principal_id
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing in .env: {', '.join(missing)}")

utils.print_ok('Notebook initialized')
utils.print_info(f"Resource Group: {resource_group_name}")
utils.print_info(f"Location: {resource_group_location}")
utils.print_info(f"Search Service: {search_service_name}")
utils.print_info(f"Storage Account: {storage_account_name}")
utils.print_info(f"Blueprint Principal: {blueprint_principal_id[:8]}...")

✅ Notebook initialized ⌚ 22:14:51.339278 
👉🏽 Resource Group: rg-agent-identity-sandbox
👉🏽 Location: eastus
👉🏽 Search Service: a365-search-tlb6wxkoo7zkk
👉🏽 Storage Account: a365sa6uuruydd4tej6
👉🏽 Blueprint Principal: 7eecd5ce...


In [2]:
# Verify Azure CLI is accessible and authenticate
test_az = utils.run("az --version", print_command_to_run=False, print_output=False)
if not test_az.success:
    utils.print_error("Azure CLI not found. Run in terminal: which az")
    utils.print_info("Then add that path to cell 1 above")
    raise RuntimeError("Cannot find az CLI")

utils.print_ok("Azure CLI is accessible")

# Authenticate with interactive browser login
try:
    token = credential.get_token('https://management.azure.com/.default')
    utils.print_ok("Azure authentication successful")
    utils.print_info(f"Authenticated with your Azure deployment identity credentials")
    
    # Get subscription ID from az account
    if not subscription_id:
        output = utils.run("az account show --query id --output tsv", print_command_to_run=False)
        if output.success:    
            subscription_id = output.text.strip()    
            utils.print_info(f"Subscription: {subscription_id}")
        else:
            utils.print_warning("Could not retrieve subscription from az CLI, but will continue")
except Exception as e:
    utils.print_error(f"Azure authentication failed: {str(e)}")
    utils.print_info("Interactive browser login is required. Check your browser for the login prompt.")
    raise

✅ Azure CLI is accessible ⌚ 22:14:54.787405 
✅ Azure authentication successful ⌚ 22:15:02.430834 
👉🏽 Authenticated with your Azure deployment identity credentials


<a id='1'></a>
## 1️⃣ Create deployment using 🦾 Bicep

This lab uses [Bicep](https://learn.microsoft.com/azure/azure-resource-manager/bicep/overview?tabs=bicep) to declaratively define all resources. The consolidated [azure-resources.bicep](azure-resources.bicep) deploys Search, Foundry, shared storage, and RBAC. If you prefer subscription-scope creation (with RG tagging), use [main.bicep](main.bicep) instead.

**Defaults:** The template generates deterministic, unique names (using `uniqueString`) for all resources and defaults the location to the resource group location. Leave the related `.env` entries empty to let the template pick those safe defaults; set them only when you need specific names/regions.

In [3]:
# Define Bicep deployment parameters for subscription-scope deployment (main.bicep)
deployment_params = {
    "blueprintPrincipalId": {"value": blueprint_principal_id},
    "resourceGroupName": {"value": resource_group_name},
    "location": {"value": resource_group_location},
    # Optional: add extra tags for the RG (SecurityControl: Ignore is set in main.bicep by default)
    "extraTags": {"value": {"source": "agent365"}}
}

deployment_name = "agent365-main"

# Check if deployment already exists and succeeded
existing = utils.run(
    f"az deployment sub show --name {deployment_name} --query properties.provisioningState --output tsv",
    print_command_to_run=False,
    print_output=False
)

if existing.success and existing.text.strip() == "Succeeded":
    utils.print_ok(f"Deployment '{deployment_name}' already exists and succeeded - skipping create")
else:
    # Write parameters to temp file
    with open('deployment-params.json', 'w') as f:
        json.dump({"$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#", 
                   "contentVersion": "1.0.0.0", "parameters": deployment_params}, f, indent=2)

    # Deploy subscription-scope template which creates RG and nested resources
    output = utils.run(
        f"az deployment sub create --name {deployment_name} --location {resource_group_location} --template-file main.bicep --parameters deployment-params.json",
        f"Deployment '{deployment_name}' succeeded",
        f"Deployment '{deployment_name}' failed"
    )

✅ Deployment 'agent365-main' already exists and succeeded - skipping create ⌚ 22:15:04.758161 


In [4]:
# Retrieve deployment outputs from subscription-scope deployment
output = utils.run(
    f"az deployment sub show --name {deployment_name} --query properties.outputs --output json",
    f"Retrieved deployment: {deployment_name}",
    f"Failed to retrieve deployment: {deployment_name}"
)

if output.success and output.json_data:
    raw = output.json_data
    outputs = raw if isinstance(raw, dict) else {}

    def get_output(key, label):
        val = outputs.get(key, {})
        if isinstance(val, dict):
            val = val.get('value', None)
        if val:
            utils.print_info(f"{label}: {val}")
        else:
            utils.print_warning(f"{label} not found in deployment outputs")
        return val

    endpoint = get_output('searchEndpoint', 'Search Endpoint')
    search_principal_id = get_output('searchPrincipalId', 'Search Managed Identity')
    search_service_name = get_output('searchServiceName', 'Search Service Name')
    storage_endpoint = get_output('storageEndpoint', 'Storage Endpoint')
    storage_account_name = get_output('storageAccountName', 'Storage Account')
    us_container = get_output('usContainerName', 'US Container')
    apac_container = get_output('apacContainerName', 'APAC Container')
    foundry_container = get_output('foundryContainerName', 'Foundry Container')
    ai_foundry_name = get_output('aiFoundryName', 'AI Foundry Name')
    ai_foundry_endpoint = get_output('aiFoundryEndpoint', 'AI Foundry Endpoint')
    ai_project_name = get_output('aiProjectName', 'AI Project Name')
else:
    utils.print_error("Deployment outputs missing; check az deployment status or parameters.")
    raise RuntimeError("Unable to retrieve deployment outputs")


⚙️ Running: az deployment sub show --name agent365-main --query properties.outputs --output json 
✅ Retrieved deployment: agent365-main ⌚ 22:15:06.241853 :1s]
👉🏽 Search Endpoint: https://a365-search-6uuruydd4tej6.search.windows.net
👉🏽 Search Managed Identity: 2e4a37c9-843e-4129-b5ca-73a73d82e7d0
👉🏽 Search Service Name: a365-search-6uuruydd4tej6
👉🏽 Storage Endpoint: https://a365sa6uuruydd4tej6.blob.core.windows.net/
👉🏽 Storage Account: a365sa6uuruydd4tej6
👉🏽 US Container: agents-us-data
👉🏽 APAC Container: agents-apac-data
👉🏽 Foundry Container: foundry-data
👉🏽 AI Foundry Name: aifz6vuv4jvgmejw
👉🏽 AI Foundry Endpoint: https://aifz6vuv4jvgmejw.openai.azure.com/
👉🏽 AI Project Name: agent365-project


<a id='2'></a>
## 2️⃣ Get Search Admin Key

Retrieve the admin key for data plane operations (creating indices, uploading documents).

In [5]:
# Retrieve the Search admin key
output = utils.run(
    f"az search admin-key show --resource-group {resource_group_name} --service-name {search_service_name} --query primaryKey --output tsv",
    "Admin key retrieved",
    "Failed to retrieve admin key"
)

if output.success:
    api_key = output.text.strip()
    utils.print_info(f"Key: ****{api_key[-4:]}")

⚙️ Running: az search admin-key show --resource-group rg-agent-identity-sandbox --service-name a365-search-6uuruydd4tej6 --query primaryKey --output tsv 
✅ Admin key retrieved ⌚ 22:15:07.713407 :1s]
👉🏽 Key: ****NeY7


<a id='3'></a>
## 3️⃣ Create Search Indices

Now that the search service exists, let's create the indices for our demo.

In [6]:
# Create two indices: agents-us and agents-apac
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, CorsOptions
)

index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

def ensure_index(name: str):
    fields = [
        SimpleField(name='id', type='Edm.String', key=True),
        SearchableField(name='title', type='Edm.String'),
        SearchableField(name='content', type='Edm.String'),
        SimpleField(name='region', type='Edm.String')
    ]
    index = SearchIndex(name=name, fields=fields, cors_options=CorsOptions(allowed_origins=['*']))
    try:
        index_client.create_index(index)
        utils.print_ok(f'Created index: {name}')
    except Exception as e:
        utils.print_warning(f'Index {name} may already exist: {e}')

ensure_index('agents-us')
ensure_index('agents-apac')

⚠️ Index agents-us may already exist: (ResourceNameAlreadyInUse) Cannot create index 'agents-us' because it already exists.
Code: ResourceNameAlreadyInUse
Message: Cannot create index 'agents-us' because it already exists.
Exception Details:	(CannotCreateExistingIndex) Cannot create index 'agents-us' because it already exists.
	Code: CannotCreateExistingIndex
	Message: Cannot create index 'agents-us' because it already exists. ⌚ 22:15:08.242195 
⚠️ Index agents-apac may already exist: (ResourceNameAlreadyInUse) Cannot create index 'agents-apac' because it already exists.
Code: ResourceNameAlreadyInUse
Message: Cannot create index 'agents-apac' because it already exists.
Exception Details:	(CannotCreateExistingIndex) Cannot create index 'agents-apac' because it already exists.
	Code: CannotCreateExistingIndex
	Message: Cannot create index 'agents-apac' because it already exists. ⌚ 22:15:08.305927 


In [7]:
# Upload 2-3 docs per index
from azure.search.documents import SearchClient

def seed_index(name: str, docs: list):
    client = SearchClient(endpoint=endpoint, index_name=name, credential=AzureKeyCredential(api_key))
    result = client.upload_documents(documents=docs)
    succeeded = sum(1 for r in result if r.succeeded)
    utils.print_ok(f'Uploaded {succeeded}/{len(docs)} docs to {name}')

seed_index('agents-us', [
    {'id': 'us-1', 'title': 'US FAQ', 'content': 'Shipping policy for US region', 'region': 'US'},
    {'id': 'us-2', 'title': 'US Returns', 'content': 'Return window and process in US', 'region': 'US'},
    {'id': 'us-3', 'title': 'US Taxes', 'content': 'Sales tax handling for US orders', 'region': 'US'},
])

seed_index('agents-apac', [
    {'id': 'apac-1', 'title': 'APAC FAQ', 'content': 'Shipping policy for APAC region', 'region': 'APAC'},
    {'id': 'apac-2', 'title': 'APAC Returns', 'content': 'Return window and process in APAC', 'region': 'APAC'},
    {'id': 'apac-3', 'title': 'APAC Taxes', 'content': 'GST/VAT handling for APAC orders', 'region': 'APAC'},
])

# Persist deployment outputs to .env for notebooks 06-08
env_updates = {
    'AZURE_RESOURCE_GROUP': resource_group_name,
    'AZURE_SEARCH_ENDPOINT': endpoint,
    'AZURE_SEARCH_SERVICE_NAME': search_service_name,
    'AZURE_STORAGE_ACCOUNT_NAME': storage_account_name,
    'AI_FOUNDRY_NAME': ai_foundry_name,
    'AI_FOUNDRY_ENDPOINT': ai_foundry_endpoint,
    'AZURE_OPENAI_ENDPOINT': ai_foundry_endpoint,
    'AI_PROJECT_NAME': ai_project_name,
    'FOUNDRY_STORAGE_ACCOUNT': storage_account_name,
    'FOUNDRY_CONTAINER': foundry_container,
}

# Read existing .env and update/append values
env_path = '.env'
existing_lines = []
if os.path.exists(env_path):
    with open(env_path, 'r') as f:
        existing_lines = f.readlines()

# Track which keys we've updated
updated_keys = set()
output_lines = []
for line in existing_lines:
    key = line.split('=')[0].strip() if '=' in line else ''
    if key in env_updates and env_updates[key]:
        output_lines.append(f"{key}={env_updates[key]}\n")
        updated_keys.add(key)
    else:
        output_lines.append(line)

# Append new keys that weren't in the file
for key, value in env_updates.items():
    if key not in updated_keys and value:
        output_lines.append(f"{key}={value}\n")

with open(env_path, 'w') as f:
    f.writelines(output_lines)

utils.print_ok(f'Updated .env with deployment outputs for notebooks 06-08')
utils.print_info(f"Keys persisted: {', '.join(k for k in env_updates.keys() if env_updates[k])}")

✅ Uploaded 3/3 docs to agents-us ⌚ 22:15:08.556452 
✅ Uploaded 3/3 docs to agents-apac ⌚ 22:15:08.810057 
✅ Updated .env with deployment outputs for notebooks 06-08 ⌚ 22:15:08.812021 
👉🏽 Keys persisted: AZURE_RESOURCE_GROUP, AZURE_SEARCH_ENDPOINT, AZURE_SEARCH_SERVICE_NAME, AZURE_STORAGE_ACCOUNT_NAME, AI_FOUNDRY_NAME, AI_FOUNDRY_ENDPOINT, AZURE_OPENAI_ENDPOINT, AI_PROJECT_NAME, FOUNDRY_STORAGE_ACCOUNT, FOUNDRY_CONTAINER


<a id='clean'></a>
## 🗑️ Clean up resources

When you're finished with the lab, you should remove all your deployed resources from Azure to avoid extra charges and keep your Azure subscription uncluttered.

```python
output = utils.run(
    f"az group delete --name {resource_group_name} -y --no-wait",
    f"Resource group '{resource_group_name}' deletion initiated",
    f"Failed to delete resource group '{resource_group_name}'"
)
```

## Summary

✅ **What was created:**
- Azure AI Search service with managed identity
- Azure AI Foundry with project and model deployments (gpt-4o, text-embedding-3-large)
- Shared storage account with containers (`agents-us-data`, `agents-apac-data`, `foundry-data`)
- Search indices with sample documents
- RBAC: Search service → Storage (for data source indexing)
- RBAC: AI Project → Storage (for foundry operations)
- RBAC: Blueprint principal → Search & Storage

**Next steps:**
- **[06-search-rbac-demo.ipynb](./06-search-rbac-demo.ipynb)**: Test index-scoped RBAC and selective agent access
- **[08-foundry-iq-agent-framework.ipynb](./08-foundry-iq-agent-framework.ipynb)**: Use the deployed Foundry resources for agent development